# SYNAPSE fixed-mixture strength sensitivity

Select an **A100**. Compare alpha 0/0.001/0.01/0.1/1.0 with eleven fixed mixtures at two teacher states and two fixed A/M pairs. Native and uniform controls are repeated. No generator fitting, B/D/test evaluation, or original-pilot rerun.

[Protocol](https://github.com/akashm776/SYNAPSE/blob/main/docs/STRENGTH_CHECK.md). This measures sensitivity, not held-out utility, and selects no winning alpha or recipe. A100 duration is unmeasured. The 30-minute per-invocation budget is checked between groups; it is not a hard timeout. Rerun after a budget pause.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO = Path('/content/SYNAPSE')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/akashm776/SYNAPSE.git', str(REPO)], check=True)
CODE_REF = ''  # On resume, enter the exact revision printed by your FIRST setup.
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REF or 'origin/main'], check=True)
print('Record strength revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)+'[llm,test]', 'numpy==2.3.5', 'accelerate==1.2.1'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); assert 'A100' in torch.cuda.get_device_name()"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/SynLess/runs/llm_qwen_v2')
OUTPUT = Path('/content/drive/MyDrive/SYNAPSE/diagnostics/qwen_strength_v1')
CONFIG = REPO / 'configs/strength_qwen_a100.json'
assert (SOURCE / 'data.json').exists(), 'Use the complete original Drive run, not the public metric archive'
print('Read-only source:', SOURCE)
print('Separate strength output:', OUTPUT)
print(CONFIG.read_text())


## Run or resume

Each completed state/pair/alpha group is saved. An interrupted group replays. After a runtime restart, restore the recorded CODE_REF and environment first. Do not run two sessions into the same folder. Logs are streamed below and appended to console.log.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'synapse.strength', '--source', str(SOURCE), '--output', str(OUTPUT), '--config', str(CONFIG)]
with (OUTPUT / 'console.log').open('a') as log:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
if returncode:
    raise RuntimeError(f'Diagnostic exited {returncode}; inspect the traceback above or {OUTPUT / "console.log"}')
status = json.loads((OUTPUT / 'strength_manifest.json').read_text())['status']
print('Strength status:', status)
if status == 'complete':
    print((OUTPUT / 'STRENGTH_REPORT.md').read_text())
else:
    print('Rerun this cell to resume; the diagnostic is not complete yet.')


## Share results

After completion share STRENGTH_REPORT.md, strength_summary.json, strength_manifest.json, and model_buffers.json. Per-group JSON files are the resumable work; no model checkpoints are generated. Preserve your original pilot and earlier diagnostic folders. Do not run the original test report again.
